# 多重比較検定をしてみよう①Turkey検定

In [ ]:
#Tukey検定のコードです。
import pandas as pd
import io
import seaborn as sns
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt

# 1. Excelからコピーしたデータをここに貼り付ける（例：3列分）
# ※ 以下の三連引用符の中に、Excelの列（タイトル行含む）をコピペしてください
csv_data = """
year1	year2	year3
data1	data2	data3
""" # ← 赤文字を消して、ここに実際のデータを貼り付け！

# 2. データの読み込みと整形
df = pd.read_csv(io.StringIO(csv_data.strip()), sep='\t')
df_melted = df.melt(var_name='Year', value_name='Temp').dropna()

# 3. Tukeyの多重比較を実行
tukey = pairwise_tukeyhsd(endog=df_melted['Temp'],     # データの値
                          groups=df_melted['Year'],    # グループ分け
                          alpha=0.05)                  # 有意水準

# 4. 結果の表示
print(tukey)

# 5. 結果をグラフで確認（どのペアに差があるか視覚化）

# グラフのサイズを設定
plt.figure(figsize=(8, 6))

# 箱ひげ図（Boxplot）を描画
# xにグループ（Year）、yに値（Temp）を指定します
sns.boxplot(data=df_melted, x='Year', y='Temp', hue='Year', palette='Set2', legend=False)

# 生のデータ点も重ねて表示（スウォームプロット）
sns.swarmplot(data=df_melted, x='Year', y='Temp', color='.25')

plt.title("Temperature Distribution by Year")
plt.xlabel("Year")
plt.ylabel("Temperature")
plt.show()

# 多重比較検定をしてみよう②Dunnett検定

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import io

# 1. Excelからコピーしたデータをここに貼り付ける
csv_data = """
year1 year2 year3
data1 data2 data3
"""

# 2. データの読み込みと整形
# sep='\s+' は「1つ以上の空白（スペースやタブ）」を区切り文字にする設定です
df = pd.read_csv(io.StringIO(csv_data.strip()), sep=r'\s+')

# ワイド形式からロング形式（縦長）に変換
df_melted = df.melt(var_name='Year', value_name='Temp').dropna()

# 3. ダネット検定の実行
# グループごとにデータをリスト化します
unique_years = df_melted['Year'].unique()
groups = [df_melted[df_melted['Year'] == y]['Temp'] for y in unique_years]

control_group = groups[0]  # 一番左の列（1980）をコントロールにします
other_groups = groups[1:]  # 残りの列（1988, 2025）と比較します

res = stats.dunnett(*other_groups, control=control_group)

# 4. 結果の表示
print(f"--- Dunnett Test Results (Control: {unique_years[0]}) ---")
print(res)

# 5. グラフの作成
plt.figure(figsize=(8, 6))
# 箱ひげ図を描くときは縦長にした df_melted を使うのが楽です
df_melted.boxplot(column='Temp', by='Year', grid=False)
plt.title('Dunnett Test: Comparison to 1980')
plt.ylabel('Temperature')
plt.suptitle('')
plt.show()



In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import io

# 1. Excelからコピーしたデータをここに貼り付ける
csv_data = """
year1 year2 year3
data1 data2 data3
"""

# 2. データの読み込みと整形
df = pd.read_csv(io.StringIO(csv_data.strip()), sep=r'\s+')

# ワイド形式からロング形式（縦長）に変換
df_melted = df.melt(var_name='Year', value_name='Temp').dropna()

# 3. ダネット検定の実行
# グループごとにデータをリスト化します
unique_years = df_melted['Year'].unique()
groups = [df_melted[df_melted['Year'] == y]['Temp'] for y in unique_years]

control_group = groups[0]  # 一番左の列（1980）をコントロールにします
other_groups = groups[1:]  # 残りの列（1988, 2025）と比較します

res = stats.dunnett(*other_groups, control=control_group)

# 4. 結果を分かりやすく表示
print(f"--- Dunnett Test Results (Control: {unique_years[0]}) ---")
print(f"{'Comparison':<20} {'Statistic':>10} {'p-value':>10} {'Lower CI':>10} {'Upper CI':>10}")
print("-" * 65)

# 他のグループ（1988, 2025）の名前をループで回して表示
conf = res.confidence_interval()

for i, year in enumerate(unique_years[1:]):
    p_val = res.pvalue[i]
    stat = res.statistic[i]
    low = conf.low[i]
    high = conf.high[i]

    comparison = f"{unique_years[0]} vs {year}"
    print(f"{comparison:<20} {stat:>10.3f} {p_val:>10.4f} {low:>10.3f} {high:>10.3f}")